In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_recall_fscore_support
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn

In [3]:
df = pd.read_csv("../data/creditcard.csv")

scaler = StandardScaler()
df["Amount_scaled"] = scaler.fit_transform(df[["Amount"]])
df["Time_scaled"]   = scaler.fit_transform(df[["Time"]])

features = [f"V{i}" for i in range(1, 29)] + ["Amount_scaled", "Time_scaled"]
X = df[features].values
y = df["Class"].values

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

fraud_count  = y_train.sum()
normal_count = (y_train == 0).sum()
ratio = normal_count / fraud_count

print(f"Train — Normal: {normal_count} | Fraud: {fraud_count} | Ratio: {ratio:.1f}x")


Train — Normal: 227451 | Fraud: 394 | Ratio: 577.3x


Approach 1 - Class Weighting

In [5]:
print("\n── Approach 1: Class Weighting ──")
mlflow.set_experiment("imbalance-comparison")

with mlflow.start_run(run_name="xgb_class_weight"):
    clf_w = XGBClassifier(
        scale_pos_weight=ratio,   # this is the key line
        n_estimators=100,
        random_state=42,
        eval_metric="aucpr",
        verbosity=0
    )
    clf_w.fit(X_train, y_train)
    y_pred_w = clf_w.predict(X_test)

    p, r, f, _ = precision_recall_fscore_support(y_test, y_pred_w, pos_label=1, average="binary")
    print(classification_report(y_test, y_pred_w, target_names=["Normal", "Fraud"]))

    mlflow.log_params({"method": "class_weight", "ratio": ratio})
    mlflow.log_metrics({"fraud_precision": p, "fraud_recall": r, "fraud_f1": f})


2026/06/27 00:54:27 INFO mlflow.tracking.fluent: Experiment with name 'imbalance-comparison' does not exist. Creating a new experiment.



── Approach 1: Class Weighting ──
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.88      0.85      0.86        98

    accuracy                           1.00     56962
   macro avg       0.94      0.92      0.93     56962
weighted avg       1.00      1.00      1.00     56962



Approach 2: SMOTE

In [6]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"After SMOTE — Normal: {(y_train_sm==0).sum()} | Fraud: {y_train_sm.sum()}")

with mlflow.start_run(run_name="xgb_smote"):
    clf_s = XGBClassifier(
        n_estimators=100,
        random_state=42,
        eval_metric="aucpr",
        verbosity=0
    )
    clf_s.fit(X_train_sm, y_train_sm)
    y_pred_s = clf_s.predict(X_test)

    p, r, f, _ = precision_recall_fscore_support(y_test, y_pred_s, pos_label=1, average="binary")
    print(classification_report(y_test, y_pred_s, target_names=["Normal", "Fraud"]))

    mlflow.log_params({"method": "smote"})
    mlflow.log_metrics({"fraud_precision": p, "fraud_recall": r, "fraud_f1": f})

After SMOTE — Normal: 227451 | Fraud: 227451
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.73      0.89      0.80        98

    accuracy                           1.00     56962
   macro avg       0.87      0.94      0.90     56962
weighted avg       1.00      1.00      1.00     56962

